In [4]:
#1
#импорт файла
# Импортируем библиотеку os для работы с путями и размерами файлов.
import os

# Импортируем модуль files для ручной загрузки файла из Windows.
from google.colab import files

# Указываем ожидаемое имя исходного файла.
имя_файла = "исходные_данные.txt"

# В Colab файлы обычно находятся в папке /content.
путь_к_файлу = os.path.join("/content", имя_файла)

# Если файла нет, предлагаем загрузить его с компьютера.
if not os.path.isfile(путь_к_файлу):
    print("Выберите исходный файл на компьютере.")
    загруженные = files.upload()

    # Проверяем, что среди загруженных файлов есть нужный.
    if имя_файла not in загруженные:
        raise FileNotFoundError(
            "Не найден файл с именем исходные_данные.txt"
        )

# Получаем размер загруженного файла в байтах.
размер = os.path.getsize(путь_к_файлу)

# Размер полного исходного файла из предыдущего этапа.
ожидаемый_размер = 132960755

print("Файл:", путь_к_файлу)
print("Размер:", размер, "байт")

# Не разрешаем продолжать работу с неполным файлом.
if размер != ожидаемый_размер:
    raise ValueError(
        "Размер файла не совпадает с исходным. "
        "Проверьте загрузку."
    )

print("Проверка размера пройдена.")

Выберите исходный файл на компьютере.


Saving исходные_данные.txt to исходные_данные.txt
Файл: /content/исходные_данные.txt
Размер: 132960755 байт
Проверка размера пройдена.


In [5]:
#2
#повторяем шаг из этапа 2 и создаем рабочую таблицу
# pandas — библиотека для работы с таблицами.
import pandas as pd

# NumPy понадобится для числовых вычислений.
import numpy as np

# Читаем таблицу без преобразования исходных значений.
# sep=";" указывает разделитель столбцов.
# dtype=str сохраняет значения в виде текста.
# keep_default_na=False не превращает пустые строки в NaN.
исходная_таблица = pd.read_csv(
    путь_к_файлу,
    sep=";",
    encoding="utf-8",
    dtype=str,
    keep_default_na=False
)

# Соединяем дату и время через один пробел.
дата_и_время = (
    исходная_таблица["Date"]
    + " "
    + исходная_таблица["Time"]
)

# Преобразуем текст в тип datetime.
# %d — день, %m — месяц, %Y — год из четырёх цифр.
# %H — часы, %M — минуты, %S — секунды.
# errors="coerce": неправильные даты становятся NaT.
время = pd.to_datetime(
    дата_и_время,
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
)

# Создаём отдельную рабочую таблицу.
# Исходная таблица остаётся неизменённой.
рабочая_таблица = pd.DataFrame({
    "datetime": время.copy()
})

# Переводим основную переменную в числовой формат.
# Знак ? и пустые значения превратятся в NaN.
рабочая_таблица["Global_active_power"] = pd.to_numeric(
    исходная_таблица["Global_active_power"],
    errors="coerce"
)

# Считаем даты, которые отсутсуют (невозможно распознать).
ошибки_дат = int(время.isna().sum())

# ищем повторяющиеся временные метки.
# keep="first" отмечает только повторные экземпляры.
повторы = int(время.duplicated(keep="first").sum())

# Вычисляем разницу между соседними датами
# в первоначальном порядке строк.
разница_времени = время.diff()

# Отрицательная разница означает движение времени назад.
обратные_переходы = int(
    (разница_времени < pd.Timedelta(0)).sum()
)

# Проверяем ожидаемую минутную сетку.
# Начало и конец определяем по корректным датам.
временные_метки = pd.DatetimeIndex(время.dropna())

ожидаемые_метки = pd.date_range(
    start=временные_метки.min(),
    end=временные_метки.max(),
    freq="min"
)

# unique() удаляет повторы только из набора меток
# для проверки; строки исходной таблицы не удаляются.
уникальные_метки = временные_метки.unique()

# difference() находит ожидаемые минуты,
# которые отсутствуют среди имеющихся меток.
отсутствующие_метки = ожидаемые_метки.difference(
    уникальные_метки
)

# Считаем наблюдения с отсутствующей мощностью.
пропуски_мощности = int(
    рабочая_таблица["Global_active_power"].isna().sum()
)

# Проверяем, что число строк рабочей таблицы не изменилось.
assert len(рабочая_таблица) == len(исходная_таблица) #assert останавливает выполнение кода при невыполнении обязательного условия

print("Количество строк:", len(рабочая_таблица))
print("Ошибки дат:", ошибки_дат)
print("Повторные временные метки:", повторы)
print("Переходы времени назад:", обратные_переходы)
print("Отсутствующие минуты:", len(отсутствующие_метки))
print("Отсутствующая мощность:", пропуски_мощности)
print("Начало:", временные_метки.min())
print("Конец:", временные_метки.max())

Количество строк: 2075259
Ошибки дат: 0
Повторные временные метки: 0
Переходы времени назад: 0
Отсутствующие минуты: 0
Отсутствующая мощность: 25979
Начало: 2006-12-16 17:24:00
Конец: 2010-11-26 21:02:00


In [6]:
#3
#таблица
# Названия семи столбцов с электрическими измерениями.
измеряемые_переменные = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

# Для каждого измерения создаём:
# 1) числовой столбец;
# 2) признак корректности измерения (True или False).
#
# Исходную таблицу не изменяем.
for имя in измеряемые_переменные:

    # Преобразуем исходное значение в число.
    # Знак "?" и пустые значения станут NaN.
    числа = pd.to_numeric(
        исходная_таблица[имя],
        errors="coerce"
    )

    # Добавляем числовую переменную в рабочую таблицу.
    рабочая_таблица[имя] = числа

    # Проверяем, что значение является конечным числом.
    # Это исключает NaN, +inf и -inf.
    корректно = числа.notna() & np.isfinite(числа)

    # Активная мощность, ток и показания подсчётчиков
    # не должны быть отрицательными.
    if имя not in ["Voltage", "Global_reactive_power"]:
        корректно = корректно & (числа >= 0)

    # Напряжение в работающей измерительной системе
    # должно быть положительным.
    if имя == "Voltage":
        корректно = корректно & (числа > 0)

    # Для реактивной мощности пока не исключаем
    # отрицательные конечные значения автоматически:
    # их физический смысл требует отдельной проверки.

    # Сохраняем результаты проверки.
    рабочая_таблица["корректно_" + имя] = корректно

# Сортируем только рабочую таблицу по времени.
# kind="stable" сохраняет исходный порядок строк
# с одинаковыми временными метками.
рабочая_таблица = рабочая_таблица.sort_values(
    "datetime",
    kind="stable"
).reset_index(drop=True)

# Независимая проверка: число строк не изменилось.
assert len(рабочая_таблица) == len(исходная_таблица)

print("Строк после сортировки:", len(рабочая_таблица))

# Выводим число отсутствующих или некорректных
# измерений отдельно по каждой переменной.
for имя in измеряемые_переменные:
    число_проблем = (
        ~рабочая_таблица["корректно_" + имя]
    ).sum()

    print(имя, ":", int(число_проблем))


Строк после сортировки: 2075259
Global_active_power : 25979
Global_reactive_power : 25979
Voltage : 25979
Global_intensity : 25979
Sub_metering_1 : 25979
Sub_metering_2 : 25979
Sub_metering_3 : 25979


In [7]:
#4
#расчет полноты

from IPython.display import display

# Получаем начало и конец исследуемого периода.
начало = рабочая_таблица["datetime"].min()
конец = рабочая_таблица["datetime"].max()

# Создаём ожидаемую сетку из всех минут.
сетка = pd.date_range(
    start=начало,
    end=конец,
    freq="min"
)

# Список столбцов с признаками корректности.
признаки = [
    "корректно_" + имя
    for имя in измеряемые_переменные
]

# Группируем данные по временной метке.
# Если одна минута встретится дважды, учитываем её один раз.
# Метод max() для True/False означает:
# минута считается корректной, если есть хотя бы
# одно корректное измерение данной переменной.
по_минутам = рабочая_таблица.groupby(
    "datetime"
)[признаки].max()

# Отдельно фиксируем присутствие временной метки.
метка_есть = pd.Series(
    True,
    index=по_минутам.index
).reindex(
    сетка,
    fill_value=False
)

# Добавляем в проверочную таблицу все ожидаемые минуты.
# Отсутствующим минутам присваиваем False.
# Это не восстановление измерений, а учёт отсутствия данных.
по_минутам = по_минутам.reindex(
    сетка,
    fill_value=False
)

# Получаем список месяцев от начала до конца наблюдений.
месяцы = pd.period_range(
    start=начало.to_period("M"),
    end=конец.to_period("M"),
    freq="M"
)

# Здесь будем собирать строки итоговых таблиц.
результаты = []
результаты_по_переменным = []
числа_по_переменным = []

# Обрабатываем каждый месяц отдельно.
for месяц in месяцы:

    # Первый момент календарного месяца.
    начало_месяца = месяц.start_time

    # Последняя минута календарного месяца.
    конец_месяца = (
        (месяц + 1).start_time
        - pd.Timedelta(minutes=1)
    )

    # Границы фактического периода записи.
    # Особенно важно для декабря 2006 и ноября 2010.
    от = max(начало_месяца, начало)
    до = min(конец_месяца, конец)

    # Все ожидаемые минуты внутри периода записи.
    окно = по_минутам.loc[от:до]

    # Сколько минут в полном календарном месяце?
    ожидалось_календарно = месяц.days_in_month * 1440

    # Сколько минут ожидается внутри периода записи?
    ожидалось_в_периоде = len(окно)

    # Сколько разных временных меток действительно есть?
    представлено = int(метка_есть.loc[от:до].sum())

    # Основная анализируемая переменная.
    активная = окно["корректно_Global_active_power"]

    # Сколько минут содержат корректную активную мощность?
    корректных = int(активная.sum())

    # Сколько ожидаемых минут осталось без пригодного
    # измерения активной мощности?
    проблемных = ожидалось_в_периоде - корректных

    # Полнота внутри фактического периода наблюдения.
    полнота = 100 * корректных / ожидалось_в_периоде

    # Полнота относительно всего календарного месяца.
    полнота_календарная = (
        100 * корректных / ожидалось_календарно
    )

    # Считаем дни с хотя бы одной минутой без
    # корректной активной мощности.
    # Краевые неполные календарные дни оцениваем
    # только в пределах периода фактической записи.
    дни = активная.groupby(активная.index.normalize())

    неполных_дней = int(
        (дни.sum() < дни.size()).sum()
    )

    # Ищем последовательные минуты без измерений.
    отсутствует = ~активная

    # Новый номер группы возникает при смене
    # True на False или False на True.
    группы = отсутствует.ne(
        отсутствует.shift()
    ).cumsum()

    # Максимальная продолжительность одного
    # непрерывного пропуска в данном месяце.
    максимальный_пропуск = int(
        отсутствует.groupby(группы).sum().max()
    )

    # Добавляем одну строку в таблицу по месяцам.
    результаты.append({
        "Месяц": str(месяц),
        "Ожидалось за полный месяц": ожидалось_календарно,
        "Ожидалось в период записи": ожидалось_в_периоде,
        "Уникальных временных меток": представлено,
        "Корректных измерений": корректных,
        "Без корректного измерения": проблемных,
        "Полнота периода, %": round(полнота, 3),
        "Полнота месяца, %": round(полнота_календарная, 3),
        "Неполных дней": неполных_дней,
        "Максимальный пропуск, мин": максимальный_пропуск,
        "Есть пропуск от 60 мин": максимальный_пропуск >= 60
    })

    # Отдельно рассчитываем полноту всех 7 переменных.
    строка_процентов = {"Месяц": str(месяц)}
    строка_количеств = {"Месяц": str(месяц)}

    for имя in измеряемые_переменные:

        n = int(окно["корректно_" + имя].sum())

        строка_процентов[имя] = round(
            100 * n / ожидалось_в_периоде,
            3
        )

        строка_количеств[имя] = n

    результаты_по_переменным.append(строка_процентов)
    числа_по_переменным.append(строка_количеств)

# Создаём итоговые таблицы.
таблица_полноты = pd.DataFrame(результаты)

полнота_переменных = pd.DataFrame(
    результаты_по_переменным
)

количества_переменных = pd.DataFrame(
    числа_по_переменным
)

# Контрольные проверки.
assert таблица_полноты["Ожидалось в период записи"].sum() == len(сетка)

assert таблица_полноты["Уникальных временных меток"].sum() == (
    рабочая_таблица["datetime"].nunique()
)

# Показываем все месяцы.
pd.set_option("display.max_rows", 60)

print("ТАБЛИЦА ПОЛНОТЫ ПО МЕСЯЦАМ")
display(таблица_полноты)

print("\nПОЛНОТА СЕМИ ПЕРЕМЕННЫХ, %")
display(полнота_переменных)

# Проверяем различия по исходным количествам, а не по округлённым процентам.
различия = (
    количества_переменных[измеряемые_переменные]
    .nunique(axis=1) > 1
)

print(
    "\nМесяцев с разной полнотой переменных:",
    int(различия.sum())
)

print(
    "Всего минут без корректной активной мощности:",
    int(таблица_полноты["Без корректного измерения"].sum())
)

print("Количество месяцев:", len(таблица_полноты))


ТАБЛИЦА ПОЛНОТЫ ПО МЕСЯЦАМ


,Месяц,Ожидалось за полный месяц,Ожидалось в период записи,Уникальных временных меток,Корректных измерений,Без корректного измерения,"Полнота периода, %","Полнота месяца, %",Неполных дней,"Максимальный пропуск, мин",Есть пропуск от 60 мин
0,2006-12,44640,21996,21996,21992,4,99.982,49.265,2,2,False
1,2007-01,44640,44640,44640,44638,2,99.996,99.996,2,1,False
2,2007-02,40320,40320,40320,40318,2,99.995,99.995,1,2,False
3,2007-03,44640,44640,44640,44639,1,99.998,99.998,1,1,False
4,2007-04,43200,43200,43200,39477,3723,91.382,91.382,3,3723,True
5,2007-05,44640,44640,44640,44640,0,100.000,100.000,0,0,False
6,2007-06,43200,43200,43200,43157,43,99.900,99.900,5,33,False
7,2007-07,44640,44640,44640,44509,131,99.707,99.707,2,83,True
8,2007-08,44640,44640,44640,44618,22,99.951,99.951,2,21,False
9,2007-09,43200,43200,43200,43198,2,99.995,99.995,1,2,False



ПОЛНОТА СЕМИ ПЕРЕМЕННЫХ, %


,Месяц,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2006-12,99.982,99.982,99.982,99.982,99.982,99.982,99.982
1,2007-01,99.996,99.996,99.996,99.996,99.996,99.996,99.996
2,2007-02,99.995,99.995,99.995,99.995,99.995,99.995,99.995
3,2007-03,99.998,99.998,99.998,99.998,99.998,99.998,99.998
4,2007-04,91.382,91.382,91.382,91.382,91.382,91.382,91.382
5,2007-05,100.000,100.000,100.000,100.000,100.000,100.000,100.000
6,2007-06,99.900,99.900,99.900,99.900,99.900,99.900,99.900
7,2007-07,99.707,99.707,99.707,99.707,99.707,99.707,99.707
8,2007-08,99.951,99.951,99.951,99.951,99.951,99.951,99.951
9,2007-09,99.995,99.995,99.995,99.995,99.995,99.995,99.995



Месяцев с разной полнотой переменных: 0
Всего минут без корректной активной мощности: 25979
Количество месяцев: 48


In [9]:
#5
#проверка длинных пропусков

# Находим минуты без корректного измерения мощности.
нет_измерения = ~рабочая_таблица[
    "корректно_Global_active_power"
]

# Каждая смена состояния (есть измерение / нет измерения)
# начинает новую последовательность.
номера_серий = нет_измерения.ne(
    нет_измерения.shift()
).cumsum()

# Выбираем только строки с пропусками.
# В исходной таблице ничего не удаляем.
пропущенные_строки = рабочая_таблица.loc[
    нет_измерения, ["datetime"]
]

# Для каждого непрерывного участка пропусков
# находим начало, конец и количество минут.
серии_пропусков = (
    пропущенные_строки
    .groupby(номера_серий[нет_измерения])["datetime"]
    .agg(["min", "max", "size"])
    .rename(columns={
        "min": "Начало",
        "max": "Конец",
        "size": "Продолжительность, мин"
    })
    .reset_index(drop=True)
)

# Проверяем, проходит ли участок через границу месяца.
серии_пропусков["Пересекает месяцы"] = (
    серии_пропусков["Начало"].dt.to_period("M")
    != серии_пропусков["Конец"].dt.to_period("M")
)

# Независимая проверка всех пропущенных минут.
assert (
    серии_пропусков["Продолжительность, мин"].sum()
    == 25979
)

print("Всего минут без измерений: 25979")
print(
    "Число серий через границу месяцев:",
    int(серии_пропусков["Пересекает месяцы"].sum())
)

print("\nДесять самых длинных пропусков:")
display(
    серии_пропусков.sort_values(
        "Продолжительность, мин",
        ascending=False
    ).head(10)
)


Всего минут без измерений: 25979
Число серий через границу месяцев: 0

Десять самых длинных пропусков:


,Начало,Конец,"Продолжительность, мин",Пересекает месяцы
68,2010-08-17 21:02:00,2010-08-22 21:27:00,7226,False
69,2010-09-25 03:56:00,2010-09-28 19:12:00,5237,False
6,2007-04-28 00:21:00,2007-04-30 14:23:00,3723,False
49,2009-06-13 00:30:00,2009-06-15 07:34:00,3305,False
58,2010-01-12 14:53:00,2010-01-14 19:01:00,3129,False
62,2010-03-20 03:52:00,2010-03-21 13:38:00,2027,False
51,2009-08-13 05:00:00,2009-08-13 19:50:00,891,False
15,2007-07-15 16:49:00,2007-07-15 18:11:00,83,False
38,2008-12-10 10:48:00,2008-12-10 11:57:00,70,False
16,2007-07-15 18:21:00,2007-07-15 19:07:00,47,False


In [10]:

# pathlib используется для работы с именами файлов.
from pathlib import Path

# Импортируем инструмент скачивания файлов из Colab.
from google.colab import files

# Определяем имена трёх итоговых таблиц.
файл_месяцев = Path("03_полнота_по_месяцам.csv")
файл_переменных = Path("03_полнота_переменных.csv")
файл_пропусков = Path("03_серии_пропусков.csv")

# Сохраняем таблицы в CSV.
# index=False: не записываем служебные номера строк.
# sep=";" — разделитель, удобный для открытия в Excel.
# encoding="utf-8-sig" — кодировка для русских названий.
таблица_полноты.to_csv(
    файл_месяцев,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

полнота_переменных.to_csv(
    файл_переменных,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

серии_пропусков.to_csv(
    файл_пропусков,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

# Проверяем, что файлы созданы и не пустые.
for файл in [
    файл_месяцев,
    файл_переменных,
    файл_пропусков
]:
    assert файл.exists() and файл.stat().st_size > 0

    print("Сохранено:", файл.name)

# Скачиваем результаты из Colab в Windows.
for файл in [
    файл_месяцев,
    файл_переменных,
    файл_пропусков
]:
    files.download(str(файл))


Сохранено: 03_полнота_по_месяцам.csv
Сохранено: 03_полнота_переменных.csv
Сохранено: 03_серии_пропусков.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
#7
# Перечисляем утверждённые месяцы.
# Сентябрь 2008 используется сразу в двух направлениях (сохраняем его только один раз).

выбранные_месяцы = [
    "2008-01",  # Зима
    "2008-04",  # Весна
    "2008-07",  # Лето
    "2008-09",  # Осень и эталон без пропусков
    "2009-08",  # Реальные пропуски
    "2010-08"   # Длительные реальные пропуски
]

# Определяем месяц каждой записи рабочей таблицы.
# dt.to_period("M") оставляет только год и месяц.
месяц_записи = рабочая_таблица[
    "datetime"
].dt.to_period("M").astype(str)

# Создаём словарь отдельных рабочих таблиц.
# Ключ словаря — месяц, значение — его таблица.
рабочие_месяцы = {}

for месяц in выбранные_месяцы:

    # Выбираем строки нужного месяца.
    условие = месяц_записи == месяц

    # .copy() создаёт самостоятельную копию.
    # Изменения в ней не затронут общую рабочую таблицу.
    рабочие_месяцы[месяц] = (
        рабочая_таблица.loc[условие].copy()
    )

    # Получаем ожидаемое количество минут
    # из уже рассчитанной таблицы полноты.
    ожидается = int(
        таблица_полноты.loc[
            таблица_полноты["Месяц"] == месяц,
            "Ожидалось в период записи"
        ].iloc[0]
    )

    # Проверяем, что ни одна строка месяца не потеряна.
    assert len(рабочие_месяцы[месяц]) == ожидается

    print(
        месяц,
        "— строк:",
        len(рабочие_месяцы[месяц])
    )

print("\nРабочие копии успешно созданы.")


2008-01 — строк: 44640
2008-04 — строк: 43200
2008-07 — строк: 44640
2008-09 — строк: 43200
2009-08 — строк: 44640
2010-08 — строк: 44640

Рабочие копии успешно созданы.
